# Hindi ASR — Whisper + PEFT LoRA Fine-tuning on FLEURS

Trains `openai/whisper-small` with PEFT LoRA adapters on the FLEURS Hindi dataset.
- **Dataset**: `google/fleurs` `hi_in` — ~1,296 real multi-speaker Hindi speech samples
- **Method**: PEFT LoRA (r=16) on decoder attention — 1.77M trainable params (0.73%)
- **Output**: ~7 MB LoRA adapter checkpoint pushed to HuggingFace Hub
- **Runtime**: ~2–3 hours on Colab T4 GPU

**Before running**: Runtime → Change runtime type → **T4 GPU**

In [ ]:
# ── 1. Check GPU ──────────────────────────────────────────
import torch
print(f"GPU: {torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'NOT AVAILABLE — change runtime to T4'}")
print(f"CUDA: {torch.version.cuda}")
print(f"Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")
assert torch.cuda.is_available(), 'No GPU! Go to Runtime → Change runtime type → T4'

In [ ]:
# ── 2. Clone repo ─────────────────────────────────────────
!git clone https://github.com/27Kushal/Hindi-asr-whisper.git
%cd Hindi-asr-whisper

In [ ]:
# ── 3. Install dependencies (Colab-compatible) ────────────
# Pin only the critical packages; let huggingface-hub, datasets etc. float
!pip install -q \
  "transformers==4.36.2" \
  "peft==0.7.1" \
  "accelerate==0.27.2" \
  "datasets>=2.16.0" \
  "evaluate>=0.4.0" \
  "jiwer>=3.0.0" \
  "soundfile" \
  "scipy" \
  "safetensors"

import importlib, peft, transformers, datasets, evaluate
print(f"peft:         {peft.__version__}")
print(f"transformers: {transformers.__version__}")
print(f"datasets:     {datasets.__version__}")
print(f"evaluate:     {evaluate.__version__}")
print("All packages OK ✓")

In [ ]:
# ── 4. Enable FP16 for faster GPU training ────────────────
import yaml
with open('configs/config.yaml') as f:
    cfg = yaml.safe_load(f)
cfg['training']['fp16'] = True          # ~2x faster on T4
cfg['training']['per_device_train_batch_size'] = 16   # T4 can handle larger batches
cfg['training']['per_device_eval_batch_size'] = 8
cfg['training']['gradient_accumulation_steps'] = 1    # effective batch = 16
cfg['training']['eval_steps'] = 100
cfg['training']['save_steps'] = 100
cfg['training']['num_train_epochs'] = 10
with open('configs/config.yaml', 'w') as f:
    yaml.dump(cfg, f)
print("Config patched for T4 GPU ✓")
print(f"  fp16: {cfg['training']['fp16']}")
print(f"  batch size: {cfg['training']['per_device_train_batch_size']}")

In [ ]:
# ── 5. HuggingFace login (to push model later) ────────────
# Get your token from https://huggingface.co/settings/tokens
from huggingface_hub import login
login()  # Enter your HF write token when prompted

In [ ]:
# ── 6. Verify FLEURS loads correctly ──────────────────────
from datasets import load_dataset
sample_ds = load_dataset('google/fleurs', 'hi_in', split='train', streaming=True)
sample = next(iter(sample_ds))
print('FLEURS fields:', list(sample.keys()))
print('Transcription:', sample.get('transcription', '')[:80])
print('Audio sample rate:', sample['audio']['sampling_rate'])
print('FLEURS OK ✓')

In [ ]:
# ── 7. (Optional) Mode A: Base Whisper zero-shot baseline ───
# Takes ~5-10 min. You can SKIP this cell and go straight to cell 8.
# It only establishes the untuned baseline for comparison.
import subprocess, json, sys

print('Running zero-shot baseline... (press Stop to skip, then run cell 8)')
result = subprocess.run(
    [sys.executable, 'scripts/run_ablation.py', '--config', 'configs/config.yaml', '--modes', 'A'],
    capture_output=False
)
try:
    with open('models/ablation/base_whisper/test_results.json') as f:
        r = json.load(f)
    print(f"Base Whisper zero-shot \u2192 WER: {r['test_wer']:.4f}, CER: {r['test_cer']:.4f}")
except FileNotFoundError:
    print('Baseline results not saved (run was interrupted or skipped).')
    print('➡ Continue to cell 8 to start LoRA training.')

In [ ]:
# ── 8. Run Mode C: LoRA PEFT training (main run) ──────────
# ~2–3 hours on T4 GPU
!CUDA_VISIBLE_DEVICES=0 python train.py \
    --config configs/config.yaml \
    2>&1 | tee training_lora.txt
print('Training complete ✓')

In [ ]:
# ── 9. Print LoRA training results ────────────────────────
import json
with open('models/whisper-lora-hindi/test_results.json') as f:
    r = json.load(f)
print('=== LoRA PEFT Results ===')
print(f"  WER:              {r['test_wer']:.4f}")
print(f"  CER:              {r['test_cer']:.4f}")
print(f"  Trainable params: {r['trainable_params']:,} ({100*r['trainable_params']/r['total_params']:.2f}%)")
print(f"  Checkpoint size:  {r['checkpoint_size_bytes']/1e6:.1f} MB")
print(f"  Training time:    {r['training_time_sec']/3600:.2f} hours")

In [ ]:
# ── 10. (Optional) Run Mode B: Encoder-frozen ablation ────
# ~2–3 hours — comparison baseline. Skip if short on time.
!CUDA_VISIBLE_DEVICES=0 python train.py \
    --config configs/config.yaml \
    --no_lora \
    --output_dir_override models/ablation/encoder_frozen \
    2>&1 | tee training_frozen.txt

In [ ]:
# ── 11. Push LoRA adapter to HuggingFace Hub ──────────────
# The adapter is only ~7 MB — uploads in seconds
from huggingface_hub import HfApi

HF_REPO = 'kushalbagla/whisper-small-hindi-lora'  # your HF username/repo

api = HfApi()
api.create_repo(HF_REPO, exist_ok=True)
api.upload_folder(
    folder_path='models/whisper-lora-hindi/final',
    repo_id=HF_REPO,
    repo_type='model',
)
print(f'Adapter pushed to: https://huggingface.co/{HF_REPO}')

In [ ]:
# ── 12. Final results summary ──────────────────────────────
import json, os

print('=' * 60)
print('FINAL RESULTS SUMMARY')
print('=' * 60)

for label, path in [
    ('Base Whisper (zero-shot)', 'models/ablation/base_whisper/test_results.json'),
    ('LoRA PEFT r=16 (FLEURS)', 'models/whisper-lora-hindi/test_results.json'),
]:
    try:
        with open(path) as f:
            r = json.load(f)
        wer = r.get('test_wer', r.get('wer', '?'))
        cer = r.get('test_cer', r.get('cer', '?'))
        print(f'\n{label}:')
        print(f'  WER: {wer:.4f}  CER: {cer:.4f}')
    except FileNotFoundError:
        print(f'\n{label}: not found')

total_mb = sum(
    os.path.getsize(os.path.join(root, f))
    for root, _, files in os.walk('models/whisper-lora-hindi/final')
    for f in files if f.endswith(('.bin', '.safetensors'))
) / 1e6
print(f'\nAdapter checkpoint: {total_mb:.1f} MB')
print('=' * 60)